# Imports and Configuration
- **`INPUT_PATH`**: folder containing the input data to process.
- **`OUTPUT_PATH`**: folder where results will be written.
- **`CELL_MASK_DIR`**: folder containing per-image cell masks named `<raw_stem>_full_cell.tif`. Sharpness is measured only inside this cell mask.

In [ ]:
import os
import numpy as np
import tifffile
import pandas as pd
from scipy import ndimage

INPUT_PATH = r"../data/images"
OUTPUT_PATH = r"../outputs/sharpness_with_cellmask"

# Folder containing per-image cell masks named "<raw_stem>_full_cell.tif".
CELL_MASK_DIR = r"../data/masks"
CELL_MASK_TOKEN = "_full_cell"

# If the output folder does not exist, create it automatically.
# os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Input folder    : {INPUT_PATH}")
print(f"Output folder   : {OUTPUT_PATH}")
print(f"Cell-mask folder: {CELL_MASK_DIR}")

# Metrics Definition

Define the image-quality metric used in this notebook.

- **`tenengrad_sharpness(image, mask=None, threshold=0.0)`**: Tenengrad sharpness score. The Sobel gradients are computed with `scipy.ndimage.sobel`, and the score is the mean squared gradient magnitude, optionally restricted to a `mask`. Higher values indicate a sharper image.

In [ ]:
def tenengrad_sharpness(image, mask=None, threshold=0.0):
    """Tenengrad sharpness score (mean squared Sobel gradient magnitude).

    Uses `scipy.ndimage.sobel` to compute the gradients.  Only gradient
    magnitudes above `threshold` are kept, and the score is the mean of those
    squared magnitudes.  If `mask` is given, the score is computed only over
    the masked pixels.  A higher score means a sharper image.
    """
    img = np.asarray(image, dtype=np.float64)

    gx = ndimage.sobel(img, axis=1, mode="nearest")
    gy = ndimage.sobel(img, axis=0, mode="nearest")
    g_squared = gx ** 2 + gy ** 2

    # Tenengrad keeps only gradients above the threshold
    g_squared[g_squared <= threshold ** 2] = 0.0

    values = g_squared[mask] if mask is not None else g_squared
    if values.size == 0:
        return None

    return float(np.mean(values))

In [ ]:
def load_probability(prob_path, prob_channel=0):
    """Load an ilastik probability TIFF and return the foreground-probability map (2D).

    Handles (H, W), (H, W, C), and (C, H, W) layouts.  Float values stay as-is;
    integer probability maps are rescaled to [0, 1].
    """
    prob_all = tifffile.imread(prob_path)
    print(prob_all.shape)
    prob = prob_all[:,:,prob_channel]

    # Not needed if prob is already between 0 to 1 .
    if np.issubdtype(prob_all.dtype, np.integer):
        prob = prob / float(np.iinfo(prob_all.dtype).max)

    return prob

In [ ]:
def sharpness_from_probability(raw_path, prob_path, threshold=0.75, fg_channel=0, cell_mask=None):
    """Load raw + foreground-probability map, threshold, and compute Tenengrad sharpness.

    If `cell_mask` (boolean, same shape as raw) is given, the foreground mask is
    restricted to pixels inside the cell mask before sharpness is computed.
    """
    raw = tifffile.imread(raw_path)
    fg_prob = load_probability(prob_path, prob_channel=fg_channel)
    if raw.shape != fg_prob.shape:
        raise ValueError(f"Shape mismatch: raw {raw.shape} vs prob {fg_prob.shape}")

    fg_mask = fg_prob >= threshold

    if cell_mask is not None:
        if cell_mask.shape != raw.shape:
            raise ValueError(f"Shape mismatch: raw {raw.shape} vs cell_mask {cell_mask.shape}")
        fg_mask = fg_mask & cell_mask

    return tenengrad_sharpness(raw, mask=fg_mask)

# File Discovery

Function that walks `INPUT_PATH` and lists `.tif` files, optionally pairing each one with a matching companion file.

- **`find_files(root, token=None)`**:
  - If `token` is `None`: returns every `.tif` file under `root`, paired with an empty string.
  - If `token` is e.g. `"_Probabilities"` or `"_Simple Segmentation"`: only returns raws that have a matching `<raw>_<token>.tif` companion.

- **`find_cell_mask(raw_path, cell_mask_dir, token)`**: locate the `<raw_stem><token>.tif` cell mask for a raw image.
- **`load_cell_mask(cell_mask_path)`**: load a cell mask as a boolean array (any non-zero pixel = inside cell).

In [ ]:
def find_files(root, token=None):
    """Walk `root` and return list of (raw_path, companion_path) tuples.

    If `token` is None, all .tif files are returned with an empty companion path.
    Otherwise, only raws that have a matching `<raw>_<token>.tif` companion are kept.
    """
    pairs = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not fname.lower().endswith(".tif"):
                continue
            full = os.path.join(dirpath, fname)

            if token is None:
                pairs.append((full, ""))
                continue

            # skip companion files themselves; only treat plain raws as candidates
            if token in fname:
                continue

            stem = fname.rsplit(".", 1)[0]
            companion = os.path.join(dirpath, stem + token + ".tif")
            if os.path.exists(companion):
                pairs.append((full, companion))

    return sorted(pairs)


def find_cell_mask(raw_path, cell_mask_dir, token=CELL_MASK_TOKEN):
    """Locate the cell-mask TIF for a given raw image.

    Searches `cell_mask_dir` recursively for `<raw_stem><token>.tif`.
    Returns the absolute path. Raises FileNotFoundError if missing.
    """
    if not cell_mask_dir:
        raise FileNotFoundError(
            f"CELL_MASK_DIR is not set; cannot locate cell mask for {raw_path}"
        )

    stem = os.path.splitext(os.path.basename(raw_path))[0]
    target = stem + token + ".tif"
    for dirpath, _, filenames in os.walk(cell_mask_dir):
        if target in filenames:
            return os.path.join(dirpath, target)

    raise FileNotFoundError(
        f"Cell mask '{target}' not found anywhere under {cell_mask_dir}"
    )


def load_cell_mask(cell_mask_path):
    """Load a cell-mask TIF as a boolean array (any non-zero pixel = inside cell)."""
    cm = tifffile.imread(cell_mask_path)
    if cm.ndim > 2:
        cm = cm[0] if cm.shape[0] < cm.shape[-1] else cm[..., 0]
    return cm > 0

## Sharpness from Segmentation masks (inside cell mask)

In [ ]:
FG_LABEL = 1

seg_pairs = find_files(INPUT_PATH, token="_Simple Segmentation")
print(f"Found {len(seg_pairs)} segmentation pair(s) under {INPUT_PATH}")
print(f"Foreground label = {FG_LABEL}\n")

seg_results = []

for raw_path, seg_path in seg_pairs:
    raw = tifffile.imread(raw_path)
    seg = tifffile.imread(seg_path)

    cell_mask_path = find_cell_mask(raw_path, CELL_MASK_DIR)
    cell_mask = load_cell_mask(cell_mask_path)
    if cell_mask.shape != raw.shape:
        raise ValueError(
            f"Shape mismatch for {raw_path}: raw {raw.shape} vs cell_mask {cell_mask.shape}"
        )

    fg_mask = (seg == FG_LABEL) & cell_mask
    sharpness = tenengrad_sharpness(raw, mask=fg_mask)
    
    seg_results.append({
        "raw_path": raw_path,
        "segmentation_path": seg_path,
        "cell_mask_path": cell_mask_path,
        "fg_label": FG_LABEL,
        "sharpness": sharpness,
    })

    print("--------------")
    print(raw_path)
    print(f"Cell mask: {cell_mask_path}")
    print("Sharpness (Tenengrad)", sharpness)
    print("--------------")

# Sharpness from Probability Maps (inside cell mask)

Use ilastik foreground-probability TIFFs to define the foreground mask via a threshold, restrict it to the cell mask, then compute the Tenengrad sharpness on the raw image.

Reuses `load_probability` and `sharpness_from_probability` defined earlier.

In [ ]:
THRESHOLD = 0.75
FG_CHANNEL = 0

prob_pairs = find_files(INPUT_PATH, token="_Probabilities")
print(f"Found {len(prob_pairs)} probability pair(s) under {INPUT_PATH}")

prob_results = []

for raw_path, prob_path in prob_pairs:
    cell_mask_path = find_cell_mask(raw_path, CELL_MASK_DIR)
    cell_mask = load_cell_mask(cell_mask_path)

    sharpness = sharpness_from_probability(
        raw_path, prob_path,
        threshold=THRESHOLD, fg_channel=FG_CHANNEL,
        cell_mask=cell_mask,
    )

    prob_results.append({
        "raw_path": raw_path,
        "probability_path": prob_path,
        "cell_mask_path": cell_mask_path,
        "threshold": THRESHOLD,
        "fg_channel": FG_CHANNEL,
        "sharpness": sharpness,
    })

    print("--------------")
    print(raw_path)
    print(f"Cell mask: {cell_mask_path}")
    print("Sharpness (Tenengrad)", sharpness)
    print("--------------")

In [ ]:
seg_csv_path = os.path.join(OUTPUT_PATH, "segmentation_metrics.csv")
prob_csv_path = os.path.join(OUTPUT_PATH, "probability_metrics.csv")

seg_df = pd.DataFrame(seg_results)
prob_df = pd.DataFrame(prob_results)

seg_df.to_csv(seg_csv_path, index=False)
prob_df.to_csv(prob_csv_path, index=False)

print(f"Saved segmentation metrics ({len(seg_df)} row(s)) to: {seg_csv_path}")
print(f"Saved probability  metrics ({len(prob_df)} row(s)) to: {prob_csv_path}")